In [ ]:
!pip install numpy pandas pytesseract pillow matplotlib opencv-python openpyxl tensorflow scikit-learn seaborn gensim

In [ ]:
import numpy as np
import pandas as pd
import os
import re
from pathlib import Path

# Path dataset tetap sesuai Kaggle
dataset_dir = './dataset'
image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

if not os.path.isdir(dataset_dir):
    raise FileNotFoundError(f'Dataset folder tidak ditemukan: {dataset_dir}')

print(f'Dataset folder: {dataset_dir}')


# image_paths

In [ ]:
# Batch load image (tidak perlu satu per satu)
image_paths = sorted([
    str(p) for p in Path(dataset_dir).iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
])

print(f'Total image: {len(image_paths)}')
print('Contoh file:')
for p in image_paths[:5]:
    print('-', os.path.basename(p))


# import

In [ ]:
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import cv2


# output_folder

In [ ]:
output_folder = './ocr_output'
os.makedirs(output_folder, exist_ok=True)
print(f'Output folder: {output_folder}')


# preprocess_image

In [ ]:
def preprocess_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Gagal membaca gambar: {path}')

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 5, 75, 75)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Normalisasi foreground/background supaya OCR lebih stabil
    if np.mean(thresh) < 127:
        thresh = 255 - thresh

    kernel = np.ones((1, 1), np.uint8)
    processed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    return processed


def normalize_text(text):
    text = text.replace('\r', ' ')
    text = text.replace('\n', ' ')
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_phrase(text):
    text = normalize_text(text.lower())
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def split_composition_items(composition_text):
    raw_items = re.split(r'[,;•|]', composition_text)
    items = []
    for item in raw_items:
        s = normalize_text(item)
        if len(s) >= 2:
            items.append(s)
    return items


def token_overlap_score(a, b):
    sa = set(clean_phrase(a).split())
    sb = set(clean_phrase(b).split())
    if not sa or not sb:
        return 0.0
    inter = len(sa.intersection(sb))
    return inter / max(1, min(len(sa), len(sb)))


def _ocr_lines(image_for_ocr, lang='eng', min_conf=35):
    data = pytesseract.image_to_data(image_for_ocr, lang=lang, output_type=pytesseract.Output.DICT)
    grouped = {}

    for i in range(len(data['text'])):
        word = (data['text'][i] or '').strip()
        if not word:
            continue
        try:
            conf = float(data['conf'][i])
        except Exception:
            continue
        if conf < min_conf:
            continue

        block = int(data['block_num'][i])
        par = int(data['par_num'][i])
        line = int(data['line_num'][i])
        left = int(data['left'][i])
        top = int(data['top'][i])
        width = int(data['width'][i])
        height = int(data['height'][i])

        key = (block, par, line)
        grouped.setdefault(key, []).append({
            'word': word,
            'left': left,
            'top': top,
            'right': left + width,
            'bottom': top + height,
            'height': height,
            'conf': conf,
            'block': block,
        })

    lines = []
    for key, words in grouped.items():
        words = sorted(words, key=lambda x: x['left'])
        text = normalize_text(' '.join(w['word'] for w in words))
        if not text:
            continue
        lines.append({
            'text': text,
            'norm': clean_phrase(text),
            'left': min(w['left'] for w in words),
            'right': max(w['right'] for w in words),
            'top': min(w['top'] for w in words),
            'bottom': max(w['bottom'] for w in words),
            'height': max(w['height'] for w in words),
            'conf': float(np.mean([w['conf'] for w in words])),
            'block': words[0]['block'],
        })

    lines.sort(key=lambda x: (x['top'], x['left']))
    return lines


def parse_composition_from_layout(path, processed_img):
    # Fokus ke area komposisi berdasarkan anchor keyword agar text luar tidak ikut
    lines = _ocr_lines(processed_img, lang='eng', min_conf=35)
    if not lines:
        return ''

    anchor_pattern = re.compile(r'\b(komposisi|composition|ingredients?)\b', re.I)
    stop_pattern = re.compile(
        r'\b(informasi nilai gizi|nutrition|takaran saji|energi total|cara penyimpanan|penyimpanan|'
        r'netto|berat bersih|expired|kedaluwarsa|bpom|kode produksi|saran penyajian|perhatian)\b',
        re.I,
    )

    anchors = [ln for ln in lines if anchor_pattern.search(ln['text'])]
    if not anchors:
        # fallback minimal: gunakan OCR full text yg dinormalisasi
        full_text = pytesseract.image_to_string(processed_img, lang='eng')
        return normalize_text(full_text)

    anchor = anchors[0]
    avg_h = np.mean([ln['height'] for ln in lines]) if lines else 20
    max_gap = max(18, int(avg_h * 1.8))

    selected = []
    started = False
    prev_bottom = None

    for ln in lines:
        if ln['top'] < anchor['top']:
            continue

        # mulai saat baris anchor ditemukan
        if not started and ln is anchor:
            started = True
            selected.append(ln['text'])
            prev_bottom = ln['bottom']
            continue

        if not started:
            continue

        # stop jika lompat vertikal terlalu jauh
        if prev_bottom is not None and (ln['top'] - prev_bottom) > max_gap:
            break

        # prefer area yang sebaris/sekolom dengan anchor
        horizontal_overlap = not (ln['right'] < anchor['left'] - 100 or ln['left'] > anchor['right'] + 900)
        near_anchor_column = abs(ln['left'] - anchor['left']) <= 220
        if not (horizontal_overlap or near_anchor_column):
            continue

        if stop_pattern.search(ln['text']):
            break

        selected.append(ln['text'])
        prev_bottom = ln['bottom']

    comp = normalize_text(' '.join(selected))
    comp = re.sub(r'(?i)^\s*(komposisi|composition|ingredients?)\s*[:\-]?\s*', '', comp).strip()
    return comp


def extract_bold_phrases(path, min_conf=45):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bin_inv = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15
    )

    data = pytesseract.image_to_data(gray, lang='eng', output_type=pytesseract.Output.DICT)

    words = []
    for i in range(len(data['text'])):
        word = (data['text'][i] or '').strip()
        if not word:
            continue
        try:
            conf = float(data['conf'][i])
        except Exception:
            continue
        if conf < min_conf:
            continue

        l = int(data['left'][i]); t = int(data['top'][i])
        w = max(1, int(data['width'][i])); h = max(1, int(data['height'][i]))
        roi = bin_inv[t:t+h, l:l+w]
        ink_ratio = float(np.mean(roi > 0)) if roi.size else 0.0

        words.append({
            'word': word, 'ink': ink_ratio, 'h': h, 'left': l,
            'block': int(data['block_num'][i]), 'par': int(data['par_num'][i]), 'line': int(data['line_num'][i])
        })

    if not words:
        return []

    ink_thr = float(np.percentile([w['ink'] for w in words], 75))
    h_thr = float(np.percentile([w['h'] for w in words], 60))
    bold_words = [w for w in words if w['ink'] >= ink_thr and w['h'] >= h_thr]

    grouped = {}
    for w in bold_words:
        grouped.setdefault((w['block'], w['par'], w['line']), []).append(w)

    phrases = []
    for _, ws in grouped.items():
        ws = sorted(ws, key=lambda x: x['left'])
        phrase = normalize_text(' '.join(x['word'] for x in ws))
        if len(clean_phrase(phrase)) >= 2:
            phrases.append(phrase)

    seen = set(); out = []
    for p in phrases:
        k = clean_phrase(p)
        if k and k not in seen:
            seen.add(k); out.append(p)
    return out


def detect_bold_composition_items(path, composition_text):
    bold_phrases = extract_bold_phrases(path)
    composition_items = split_composition_items(composition_text)

    bold_items = []
    for item in composition_items:
        item_clean = clean_phrase(item)
        if not item_clean:
            continue
        for bp in bold_phrases:
            bp_clean = clean_phrase(bp)
            if not bp_clean:
                continue
            overlap = token_overlap_score(item, bp)
            contains = item_clean in bp_clean or bp_clean in item_clean
            if overlap >= 0.6 or contains:
                bold_items.append(item)
                break

    uniq = []
    seen = set()
    for x in bold_items:
        k = clean_phrase(x)
        if k and k not in seen:
            seen.add(k)
            uniq.append(x)

    label = 'unsafe' if uniq else 'safe'
    return label, uniq


In [ ]:
records = []

for idx, path in enumerate(image_paths, start=1):
    base_name = os.path.splitext(os.path.basename(path))[0]
    print(f'[{idx}/{len(image_paths)}] Processing: {base_name}')

    processed = preprocess_image(path)

    # OCR full text (raw) tetap disimpan untuk audit
    raw_text = pytesseract.image_to_string(processed, lang='eng')

    # Komposisi diambil dari section/layout agar text luar tidak ikut
    composition_text = parse_composition_from_layout(path, processed)

    txt_filename = f'{base_name}.txt'
    output_path = os.path.join(output_folder, txt_filename)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(raw_text)

    label, bold_composition_items = detect_bold_composition_items(path, composition_text)

    records.append({
        'nama_produk': base_name,
        'text': composition_text,
        'label': label,
        'alergen_dari_teks_tebal': '; '.join(bold_composition_items),
    })

df_dataset = pd.DataFrame(records)
print('\nSelesai build dataset:')
print(df_dataset.head(10))


In [ ]:
# Tampilkan dataframe final (nama produk, text/komposisi, label)
df_final = df_dataset[['nama_produk', 'text', 'label']].copy()
display(df_final.head(20))
print(f'Total baris: {len(df_final)}')


In [ ]:
# Export dataset ke CSV dan XLSX
csv_path = os.path.join(output_folder, 'dataset_komposisi_label.csv')
# csv_path = os.path.join(output_folder, 'data-mengandung.csv')
xlsx_path = os.path.join(output_folder, 'dataset_komposisi_label.xlsx')

df_final.to_csv(csv_path, index=False, encoding='utf-8')
df_final.to_excel(xlsx_path, index=False)

print('CSV :', csv_path)
print('XLSX:', xlsx_path)


## Mode Data (Pilih Satu)
Notebook ini bisa jalan dengan 2 mode data:
1. `OCR` : ambil data dari proses OCR yang baru dijalankan.
2. `CSV` : langsung pakai file CSV hasil OCR sebelumnya (lebih cepat).


In [ ]:
# =========================
# PANEL KONFIGURASI UTAMA
# =========================
# Semua pengaturan penting dikumpulkan di sini agar mudah diubah.

# Pilih sumber data:
# - 'OCR' : pakai hasil OCR dari cell sebelumnya
# - 'CSV' : langsung pakai file CSV hasil OCR (lebih cepat)
DATA_SOURCE_MODE = 'CSV'

# Jika mode CSV dipilih, isi lokasi file CSV di bawah ini
# CSV_INPUT_PATH = './ocr_output/dataset_komposisi_label.csv'
CSV_INPUT_PATH = './ocr_output/data-mengandung.csv'

# Nama kolom yang dipakai untuk training
TEXT_COL = 'text'
LABEL_COL = 'label'

# Seed agar hasil eksperimen lebih konsisten
SEED = 42

# Hyperparameter utama model
TEST_SIZE = 0.2
VOCAB_SIZE = 20000
MAX_LEN = 120
EMBED_DIM = 100
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DROPOUT_RATE_1 = 0.3
DROPOUT_RATE_2 = 0.3
DENSE_UNITS = 64
DROPOUT_RATE_DENSE = 0.2
LEARNING_RATE = 1e-4
BATCH_SIZE = 8
EPOCHS = 150

# Saklar callback training
# True  = aktif
# False = nonaktif
EARLY_STOPPING_ON = False
EARLY_STOPPING_PATIENCE = 4
LR_REDUCE_ON = True
LR_REDUCE_PATIENCE = 2

print('DATA_SOURCE_MODE :', DATA_SOURCE_MODE)
print('EARLY_STOPPING_ON:', EARLY_STOPPING_ON)


In [ ]:
import random
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)


In [ ]:
# ======================================
# LOAD DATA (OCR atau CSV)
# ======================================
# Tujuan: menyiapkan dataframe final untuk training

if DATA_SOURCE_MODE.upper() == 'OCR':
    # Mode OCR: gunakan dataframe yang dibuat dari proses OCR di atas
    if 'df_final' not in globals():
        raise ValueError('df_final tidak ditemukan. Jalankan cell OCR terlebih dahulu atau ganti ke mode CSV.')
    df_model_source = df_final.copy()
elif DATA_SOURCE_MODE.upper() == 'CSV':
    # Mode CSV: langsung baca file CSV hasil OCR sebelumnya
    print(f'Memuat data dari CSV: {CSV_INPUT_PATH}')
    if not os.path.exists(CSV_INPUT_PATH):
        raise FileNotFoundError(f'CSV tidak ditemukan: {CSV_INPUT_PATH}')
    df_model_source = pd.read_csv(CSV_INPUT_PATH, delimiter=';')
else:
    raise ValueError("DATA_SOURCE_MODE harus 'OCR' atau 'CSV'")

# Pastikan kolom yang dibutuhkan tersedia
for col in [TEXT_COL, LABEL_COL]:
    if col not in df_model_source.columns:
        raise ValueError(f'Kolom {col} tidak ada di data source.')

# Pembersihan data dasar
# 1) ambil kolom yang dipakai
# 2) isi nilai kosong
# 3) rapikan label
# 4) buang text kosong
# 5) simpan hanya label safe/unsafe
df_model = df_model_source[[TEXT_COL, LABEL_COL]].copy()
df_model[TEXT_COL] = df_model[TEXT_COL].fillna('').astype(str)
df_model[LABEL_COL] = df_model[LABEL_COL].fillna('').astype(str).str.strip().str.lower()
df_model = df_model[df_model[TEXT_COL].str.strip() != ''].copy()
df_model = df_model[df_model[LABEL_COL].isin(['safe', 'unsafe'])].copy()

print('Jumlah data siap model:', len(df_model))
display(df_model.head(10))


In [ ]:
# ======================================
# TEXT CLEANSING
# ======================================
def cleanse_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_model[TEXT_COL] = df_model[TEXT_COL].apply(cleanse_text)
print('Contoh setelah cleansing:')
display(df_model.head(5))


In [ ]:
# ======================================
# TRAIN / TEST SPLIT
# ======================================
label_encoder = LabelEncoder()
df_model['label_id'] = label_encoder.fit_transform(df_model[LABEL_COL])

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_model[TEXT_COL].tolist(),
    df_model['label_id'].values,
    test_size=TEST_SIZE,
    random_state=SEED
)

print('Train size:', len(X_train_text))
print('Test size :', len(X_test_text))
print('Label map :', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


In [ ]:
# ======================================
# TOKENISASI + WORD2VEC
# ======================================
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return [tok for tok in text.split() if tok]

train_tokens = [simple_tokenize(t) for t in X_train_text]

# Filtering setelah tokenizing (Sastrawi untuk bahasa Indonesia)
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
STOPWORDS = set(StopWordRemoverFactory().get_stop_words())

def filter_tokens(tokens):
    return [
        t for t in tokens
        if t not in STOPWORDS
        and len(t) >= 2
        and not t.isdigit()
    ]

train_tokens = [filter_tokens(t) for t in train_tokens]
print('Contoh token setelah filtering:')
for i in range(min(3, len(train_tokens))):
    print(f'  [{i}] {train_tokens[i][:20]}')

all_texts = X_train_text + X_test_text

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(all_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=20,
    seed=SEED
)

num_words = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
embedding_matrix = np.random.normal(scale=0.6, size=(num_words, EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = np.zeros((EMBED_DIM,), dtype=np.float32)

for word, idx in tokenizer.word_index.items():
    if idx >= num_words:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print('Tokenizer vocab:', len(tokenizer.word_index))
print('Embedding shape:', embedding_matrix.shape)


In [ ]:
# ======================================
# HYPERPARAMETER TUNING & 2 MODEL (BiLSTM + LSTM)
# ======================================

# --- Ruang hyperparameter ---
BATCH_SIZE_OPTIONS = [16, 32, 64]
LSTM_UNITS_OPTIONS = [[32, 64], [16, 32], [64, 64]]
DROPOUT_OPTIONS = [[0.1, 0.1], [0.3, 0.3], [0.0, 0.0]]
TUNING_EPOCHS_OPTIONS = [50, 75, 100]
LR_OPTIONS = [1e-4, 1e-3]
NUM_TRIALS = 10


# --- Fungsi builder model ---
def build_model(use_bidirectional, lstm_units_1, lstm_units_2, dropout_1, dropout_2, lr):
    model = Sequential()
    model.add(Embedding(
        input_dim=num_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True
    ))
    lstm_layer = Bidirectional if use_bidirectional else lambda x: x
    model.add(lstm_layer(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Dropout(dropout_1))
    model.add(lstm_layer(LSTM(lstm_units_2)))
    model.add(Dropout(dropout_2))
    model.add(Dense(DENSE_UNITS, activation='relu'))
    model.add(Dropout(DROPOUT_RATE_DENSE))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


# --- Random search ---
def tune_model(use_bidirectional, model_name):
    best_val_loss = float('inf')
    best_params = None

    for trial in range(NUM_TRIALS):
        batch_size = int(np.random.choice(BATCH_SIZE_OPTIONS))
        lstm_pair = LSTM_UNITS_OPTIONS[np.random.randint(len(LSTM_UNITS_OPTIONS))]
        dropout_pair = DROPOUT_OPTIONS[np.random.randint(len(DROPOUT_OPTIONS))]
        tuning_epochs = int(np.random.choice(TUNING_EPOCHS_OPTIONS))
        lr = float(np.random.choice(LR_OPTIONS))

        params = {
            'lstm_units_1': lstm_pair[0],
            'lstm_units_2': lstm_pair[1],
            'dropout_1': dropout_pair[0],
            'dropout_2': dropout_pair[1],
            'lr': lr,
            'batch_size': batch_size,
            'tuning_epochs': tuning_epochs,
        }

        print(f'\n=== Trial {trial+1}/{NUM_TRIALS} {model_name}: {params} ===')
        tf.random.set_seed(SEED + trial)
        m = build_model(use_bidirectional, lstm_units_1=lstm_pair[0], lstm_units_2=lstm_pair[1],
                        dropout_1=dropout_pair[0], dropout_2=dropout_pair[1], lr=lr)
        cb = [ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)]
        h = m.fit(
            X_train_pad, y_train,
            validation_data=(X_test_pad, y_test),
            epochs=tuning_epochs,
            batch_size=batch_size,
            callbacks=cb,
            verbose=0
        )
        val_loss = min(h.history['val_loss'])
        print(f'  val_loss={val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_params = params

    print(f'\n=== Best params for {model_name}: {best_params} (val_loss={best_val_loss:.4f}) ===')

    # Train final model with best params
    print(f'\n=== Training final {model_name} ===')
    tf.random.set_seed(SEED)
    final_model = build_model(
        use_bidirectional,
        lstm_units_1=best_params['lstm_units_1'],
        lstm_units_2=best_params['lstm_units_2'],
        dropout_1=best_params['dropout_1'],
        dropout_2=best_params['dropout_2'],
        lr=best_params['lr'],
    )
    callbacks = []
    if EARLY_STOPPING_ON:
        callbacks.append(EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True))
    if LR_REDUCE_ON:
        callbacks.append(ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=LR_REDUCE_PATIENCE, min_lr=1e-6))

    final_history = final_model.fit(
        X_train_pad, y_train,
        validation_data=(X_test_pad, y_test),
        epochs=EPOCHS,
        batch_size=best_params['batch_size'],
        callbacks=callbacks if len(callbacks) > 0 else None,
        verbose=1
    )
    return final_model, final_history, best_params


print('='*60)
print('HYPERPARAMETER TUNING & TRAINING: BiLSTM')
print('='*60)
model_bilstm, history_bilstm, best_params_bilstm = tune_model(True, 'BiLSTM')
model_bilstm.summary()

print('\n' + '='*60)
print('HYPERPARAMETER TUNING & TRAINING: LSTM')
print('='*60)
model_lstm, history_lstm, best_params_lstm = tune_model(False, 'LSTM')
model_lstm.summary()

tuning_results = {
    'BiLSTM': best_params_bilstm,
    'LSTM': best_params_lstm,
}
print('\nTuning results:', tuning_results)


In [ ]:
# ======================================
# TRAINING LOG FIGURE (BiLSTM & LSTM)
# ======================================
def plot_training_history(history, model_name):
    hist = pd.DataFrame(history.history)

    if {'precision', 'recall'}.issubset(hist.columns):
        hist['f1'] = 2 * (hist['precision'] * hist['recall']) / (hist['precision'] + hist['recall'] + 1e-8)
    if {'val_precision', 'val_recall'}.issubset(hist.columns):
        hist['val_f1'] = 2 * (hist['val_precision'] * hist['val_recall']) / (hist['val_precision'] + hist['val_recall'] + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hist['accuracy'], label='train_accuracy')
    if 'val_accuracy' in hist.columns: axes[0].plot(hist['val_accuracy'], label='val_accuracy')
    if 'f1' in hist.columns: axes[0].plot(hist['f1'], label='train_f1')
    if 'val_f1' in hist.columns: axes[0].plot(hist['val_f1'], label='val_f1')
    axes[0].set_title(f'{model_name} - Training Log Accuracy/F1')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Score')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(hist['loss'], label='train_loss')
    if 'val_loss' in hist.columns: axes[1].plot(hist['val_loss'], label='val_loss')
    axes[1].set_title(f'{model_name} - Training Log Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    history_csv = os.path.join(output_folder, f'training_history_{model_name.lower()}_word2vec.csv')
    hist.to_csv(history_csv, index=False)
    print(f'Training history ({model_name}) saved to:', history_csv)

plot_training_history(history_bilstm, 'BiLSTM')
plot_training_history(history_lstm, 'LSTM')


In [ ]:
# ======================================
# EVALUASI KEDUA MODEL + EXPORT TABLE
# ======================================
def evaluate_model(model, model_name, X_test, y_test, label_encoder):
    print(f'\n========== EVALUATION: {model_name} ==========')
    y_prob = model.predict(X_test).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    eval_table = pd.DataFrame([
        {'metric': 'accuracy', 'value': acc},
        {'metric': 'precision', 'value': prec},
        {'metric': 'recall', 'value': rec},
        {'metric': 'f1_score', 'value': f1},
        {'metric': 'roc_auc', 'value': auc},
    ])

    print('Evaluation Table:')
    display(eval_table)

    target_names = list(label_encoder.classes_)
    report_dict = classification_report(y_test, y_pred, target_names=target_names, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={'index': 'label'})
    print('Classification Report Table:')
    display(report_df)

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # --- ROC Curve ---
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.4f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - ROC Curve')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    pred_table = pd.DataFrame({
        'text': X_test_text,
        'label_actual': label_encoder.inverse_transform(y_test),
        'label_pred': label_encoder.inverse_transform(y_pred),
        'score_unsafe': y_prob
    })

    return eval_table, report_df, pred_table, fpr, tpr, auc


# Evaluasi BiLSTM
eval_bilstm, report_bilstm, pred_bilstm, fpr_bilstm, tpr_bilstm, auc_bilstm = \
    evaluate_model(model_bilstm, 'BiLSTM', X_test_pad, y_test, label_encoder)

# Evaluasi LSTM
eval_lstm, report_lstm, pred_lstm, fpr_lstm, tpr_lstm, auc_lstm = \
    evaluate_model(model_lstm, 'LSTM', X_test_pad, y_test, label_encoder)

# --- ROC Comparison ---
plt.figure(figsize=(8, 6))
plt.plot(fpr_bilstm, tpr_bilstm, label=f'BiLSTM (AUC = {auc_bilstm:.4f})', linewidth=2)
plt.plot(fpr_lstm, tpr_lstm, label=f'LSTM (AUC = {auc_lstm:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: BiLSTM vs LSTM')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Analisis ROC:')
if auc_bilstm > auc_lstm:
    print(f'  BiLSTM (AUC={auc_bilstm:.4f}) lebih baik dari LSTM (AUC={auc_lstm:.4f})')
elif auc_lstm > auc_bilstm:
    print(f'  LSTM (AUC={auc_lstm:.4f}) lebih baik dari BiLSTM (AUC={auc_bilstm:.4f})')
else:
    print(f'  Kedua model setara (AUC={auc_bilstm:.4f})')
print('  AUC mendekati 1.0 = klasifikasi sempurna, 0.5 = random.')

# --- Export BiLSTM ---
eval_bilstm.to_csv(os.path.join(output_folder, 'evaluation_table_bilstm_word2vec.csv'), index=False)
eval_bilstm.to_excel(os.path.join(output_folder, 'evaluation_table_bilstm_word2vec.xlsx'), index=False)
report_bilstm.to_csv(os.path.join(output_folder, 'classification_report_bilstm_word2vec.csv'), index=False)
pred_bilstm.to_csv(os.path.join(output_folder, 'predictions_test_bilstm_word2vec.csv'), index=False)

# --- Export LSTM ---
eval_lstm.to_csv(os.path.join(output_folder, 'evaluation_table_lstm_word2vec.csv'), index=False)
eval_lstm.to_excel(os.path.join(output_folder, 'evaluation_table_lstm_word2vec.xlsx'), index=False)
report_lstm.to_csv(os.path.join(output_folder, 'classification_report_lstm_word2vec.csv'), index=False)
pred_lstm.to_csv(os.path.join(output_folder, 'predictions_test_lstm_word2vec.csv'), index=False)

# --- Export perbandingan ---
comparison = pd.DataFrame([
    {
        'model': 'BiLSTM',
        'best_params': str(best_params_bilstm),
        **{row['metric']: row['value'] for _, row in eval_bilstm.iterrows()}
    },
    {
        'model': 'LSTM',
        'best_params': str(best_params_lstm),
        **{row['metric']: row['value'] for _, row in eval_lstm.iterrows()}
    },
])
comparison.to_csv(os.path.join(output_folder, 'comparison_bilstm_vs_lstm.csv'), index=False)

print('\nSaved all evaluation files.')
print('\n========== MODEL COMPARISON ==========')
display(comparison)


## Saran Hyperparameter (Praktis)
Gunakan panduan ini saat tuning:

1. `MAX_LEN`
- Jika banyak komposisi panjang terpotong, naikkan ke `150-200`.
- Jika training lambat, turunkan ke `80-100`.

2. `EMBED_DIM`
- Mulai dari `100`.
- Jika data lebih besar, bisa coba `200`.

3. `LSTM_UNITS_1` dan `LSTM_UNITS_2`
- Awal aman: `128` dan `64`.
- Jika overfitting, turunkan ke `64` dan `32`.

4. `DROPOUT_RATE_*`
- Jika train bagus tapi val jelek (overfit), naikkan dropout ke `0.4-0.5`.
- Jika model underfit, turunkan ke `0.2-0.3`.

5. `LEARNING_RATE`
- Umumnya stabil di `1e-3`.
- Jika loss naik-turun liar, turunkan ke `5e-4` atau `1e-4`.

6. `BATCH_SIZE`
- Mulai `16`.
- Jika RAM cukup, coba `32` untuk lebih cepat.

7. Early stopping
- `EARLY_STOPPING_ON = True` saat eksperimen normal.
- `False` jika ingin melihat seluruh epoch (analisis kurva training).

8. Imbalance label
- Jika `safe/unsafe` tidak seimbang, pertimbangkan `class_weight` agar F1 lebih baik.
